# Improving museum accuracy: stronger model, fine-tuning, broader style groups

Before you start: **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. About **1 hour** on a T4 (an L4 or A100 is faster).

Same data (`My Drive/colab_data.zip`), same artist-disjoint WikiArt split, same museum test set. The main numbers are for **painters the model never saw in WikiArt** (unseen artists), as in RQ1. Every choice (probe strength, blend weight, which model is "best") is made on **WikiArt val only**; museum images are never used to choose anything.

What it tests (the levers that can actually move accuracy):
1. **Stronger frozen models**: SigLIP so400m (new), plus the CLIP and DINOv2 features already on Drive from the probe notebook. Class-balanced probes, because balanced accuracy is the metric.
2. **Feature ensemble**: CLIP + DINOv2 + SigLIP features joined into one probe.
3. **Fine-tuning CLIP ViT-L/14** (last 8 of 24 blocks), starting from the style names as the classifier, then blending back towards the original weights (WiSE-FT), which is known to protect accuracy on new domains. Mirror-image test-time averaging.
4. **Final ensemble** of the fine-tuned model and the feature ensemble.
5. **Broader style groups** (an option, not a silent change; it changes the research question). Every model is also scored with the six styles merged into
   - **4 groups**: Renaissance (Northern Renaissance + Mannerism, i.e. "Late Renaissance"), Baroque (Baroque + Rococo, i.e. "Late Baroque"), Romanticism, Realism;
   - **3 eras**: Renaissance, Baroque/Rococo, 19th century.

   The grouping follows the style names and art history, not the museum results. Chance is 17% for 6 styles, 25% for 4 groups, 33% for 3 eras, so merged numbers must be read against a higher floor.

**Not tested: more epochs.** The ResNet-50 logs (`results/runs/*/history.json`) show WikiArt val balanced accuracy flat at about 55-56% from epoch 4 to 12 in all three seeds while the training loss kept falling, so more epochs would mostly memorise WikiArt. **More WikiArt data** is also not available: all 26,144 WikiArt paintings in these six styles are already used.

If Colab disconnects, just **Run all** again: features and the fine-tuned checkpoint are saved to Drive and are reused. Results go to `My Drive/wikiart_runs/improve/`. The last cells print one table to paste back.

## 0. Settings (leave as they are for the first run)

In [1]:
SEED = 1               # repeat with 2 and 3 later for the report if time allows
QUICK = False          # True = tiny run (about 10 minutes) to check that everything works
FT_EPOCHS = 4          # CLIP fine-tuning epochs (best epoch chosen on WikiArt val)
TIME_BUDGET_MIN = 28   # fine-tuning stops after the epoch that passes this many minutes
N_TRAIN_BLOCKS = 8     # CLIP blocks to fine-tune, of 24
import warnings; warnings.filterwarnings('ignore')

## 1. Connect Drive and unpack the data (about 3 minutes)

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os, time
T_START = time.time()
if not os.path.exists('/content/data/final/wikiart_split.csv'):
    !cp /content/drive/MyDrive/colab_data.zip /content/
    !cd /content && unzip -q -o colab_data.zip && rm colab_data.zip
!nvidia-smi --query-gpu=name,memory.total --format=csv

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
name, memory.total [MiB]
Tesla T4, 15360 MiB


## 2. Load the sets (same split and museum test set as the baseline)

In [3]:
import numpy as np, pandas as pd
from pathlib import Path
DATA = Path('/content/data')
PROBES = Path('/content/drive/MyDrive/wikiart_runs/probes')   # CLIP / DINOv2 features from the probe notebook
OUT = Path('/content/drive/MyDrive/wikiart_runs/improve'); OUT.mkdir(parents=True, exist_ok=True)
if QUICK: OUT = OUT / 'quick'; OUT.mkdir(exist_ok=True)

split = pd.read_csv(DATA / 'final/wikiart_split.csv')
CLASSES = sorted(split['style'].unique())
museum = pd.read_csv(DATA / 'final/museum_test.csv')
museum = museum[museum['style'].isin(CLASSES)]
sets = {s: split[split['split'] == s] for s in ('train', 'val', 'test')}
sets['met'] = museum[museum['museum'] == 'met']
sets['rijks'] = museum[museum['museum'] == 'rijks']
for k in sets:
    df = sets[k]
    sets[k] = df[df['image'].map(lambda p: (DATA / p).exists())].reset_index(drop=True)
FULL_SIZES = {k: len(v) for k, v in sets.items()}
if QUICK:
    keep = {k: df.groupby('style').head(40).index.values for k, df in sets.items()}
    sets = {k: df.loc[keep[k]].reset_index(drop=True) for k, df in sets.items()}
Y = {k: np.array([CLASSES.index(s) for s in df['style']]) for k, df in sets.items()}
EVAL = ('val', 'test', 'met', 'rijks')
print('classes:', CLASSES)
for k, df in sets.items():
    print(f'{k:<6} {len(df):>6} images')
print('Expected (full run): train 18501, val 3786, test 3857, met 937, rijks 3092')

classes: ['Baroque', 'Mannerism (Late Renaissance)', 'Northern Renaissance', 'Realism', 'Rococo', 'Romanticism']
train   18501 images
val      3786 images
test     3857 images
met       937 images
rijks    3092 images
Expected (full run): train 18501, val 3786, test 3857, met 937, rijks 3092


## 3. Metrics: six styles, plus the 4-group and 3-era options

In [4]:
from sklearn.metrics import balanced_accuracy_score, f1_score, confusion_matrix, recall_score

GROUPINGS = {
    '6 styles': {c: c for c in CLASSES},
    '4 groups': {'Northern Renaissance': 'Renaissance', 'Mannerism (Late Renaissance)': 'Renaissance',
                 'Baroque': 'Baroque+Rococo', 'Rococo': 'Baroque+Rococo',
                 'Romanticism': 'Romanticism', 'Realism': 'Realism'},
    '3 eras':   {'Northern Renaissance': 'Renaissance', 'Mannerism (Late Renaissance)': 'Renaissance',
                 'Baroque': 'Baroque+Rococo', 'Rococo': 'Baroque+Rococo',
                 'Romanticism': '19th century', 'Realism': '19th century'},
}
def collapse(y, probs, g):
    """Merge style probabilities into broader groups (sum over the styles in each group)."""
    mp = GROUPINGS[g]; labs = sorted(set(mp.values()))
    M = np.array([[1.0 * (mp[c] == l) for l in labs] for c in CLASSES])
    return np.array([labs.index(mp[CLASSES[i]]) for i in y]), probs @ M

def ece(y, probs, bins=15):
    conf, pred = probs.max(1), probs.argmax(1)
    edges = np.linspace(0, 1, bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return e

def unseen(k):
    return np.ones(len(sets[k]), bool) if k in ('train', 'val', 'test') else (sets[k]['artist_group'].values == 'unseen')

RESULTS = {}   # model name -> {set: probs (n x 6)}
def save_preds(name, probs):
    RESULTS[name] = probs
    for k, pr in probs.items():
        out = sets[k][['image', 'style']].copy(); out['pred'] = [CLASSES[i] for i in pr.argmax(1)]
        for i, c in enumerate(CLASSES): out[f'p_{c}'] = pr[:, i]
        if 'artist_group' in sets[k]: out['artist_group'] = sets[k]['artist_group'].values
        out.to_csv(OUT / f'preds_{name}_{k}.csv', index=False)

def show(name):
    pr = RESULTS[name]; line = f'{name:<20}'
    for k in EVAL:
        m = unseen(k); line += f'  {k} {100*balanced_accuracy_score(Y[k][m], pr[k][m].argmax(1)):5.1f}'
    print(line + '   (6-style bal acc; museums = unseen artists)', flush=True)

## 4. Frozen features: CLIP and DINOv2 (reused from Drive) and SigLIP (new, about 8 minutes)
Saved to Drive, so this runs only once. If the CLIP / DINOv2 files are missing they are made here too (about 8 minutes each).

In [5]:
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import AutoImageProcessor, AutoModel, CLIPModel, CLIPProcessor
dev = 'cuda'
def vec(o):   # newer transformers return an output object instead of a tensor
    return o if torch.is_tensor(o) else o.pooler_output
MODELS = {'clip': 'openai/clip-vit-large-patch14', 'dinov2': 'facebook/dinov2-large',
          'siglip': 'google/siglip-so400m-patch14-224'}   # 224 px: the images are 256 px, so the 384 px version adds time, not detail

class Images(Dataset):
    def __init__(self, paths, tf): self.paths, self.tf = paths, tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i): return self.tf(Image.open(DATA / self.paths[i]).convert('RGB'))

def feature_file(name):
    return (OUT / f'quick_features_{name}.npz') if QUICK else (PROBES / f'features_{name}.npz')

@torch.no_grad()
def extract(name):
    f = feature_file(name)
    if f.exists():
        old = dict(np.load(f))
        if all(k in old and len(old[k]) == len(sets[k]) for k in sets):
            print(name, 'features already on Drive'); return old
        print(name, 'features on Drive do not match the sets, extracting again')
    PROBES.mkdir(parents=True, exist_ok=True)
    if name == 'clip':
        model = CLIPModel.from_pretrained(MODELS[name], torch_dtype=torch.float16).to(dev).eval()
        proc = CLIPProcessor.from_pretrained(MODELS[name]).image_processor
        embed = lambda x: vec(model.get_image_features(pixel_values=x))
    elif name == 'siglip':
        model = AutoModel.from_pretrained(MODELS[name], torch_dtype=torch.float16).to(dev).eval()
        proc = AutoImageProcessor.from_pretrained(MODELS[name])
        embed = lambda x: vec(model.get_image_features(pixel_values=x))
    else:
        model = AutoModel.from_pretrained(MODELS[name], torch_dtype=torch.float16).to(dev).eval()
        proc = AutoImageProcessor.from_pretrained(MODELS[name])
        def embed(x):
            h = model(pixel_values=x).last_hidden_state
            return torch.cat([h[:, 0], h[:, 1:].mean(1)], 1)   # CLS + mean patch token
    tf = lambda img: proc(images=img, return_tensors='pt')['pixel_values'][0]
    feats, t0 = {}, time.time()
    for k, df in sets.items():
        dl = DataLoader(Images(df['image'].tolist(), tf), batch_size=64, num_workers=2)
        feats[k] = np.concatenate([embed(x.to(dev, torch.float16)).float().cpu().numpy() for x in dl])
        print(f'{name} {k:<6} {feats[k].shape}  ({(time.time()-t0)/60:.1f} min)', flush=True)
    np.savez(f, **feats)
    del model; torch.cuda.empty_cache()
    return feats

FEAT = {name: extract(name) for name in MODELS}
print(f'elapsed {(time.time()-T_START)/60:.0f} min')

clip features already on Drive
dinov2 features already on Drive
siglip features already on Drive
elapsed 0 min


## 5. Linear probes (about 5 minutes)
Class-balanced, strength chosen on WikiArt val. `clip_probe_bal` is the same CLIP probe as before, only class-balanced, so it is the fair "before" for this notebook.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

def probe(names, y_train=None, y_val=None, Cs=(0.003, 0.01, 0.03, 0.1)):
    y_train = Y['train'] if y_train is None else y_train
    y_val = Y['val'] if y_val is None else y_val
    F = {k: np.concatenate([FEAT[n][k] / np.linalg.norm(FEAT[n][k], axis=1, keepdims=True) for n in names], 1) for k in sets}
    sc = StandardScaler().fit(F['train']); F = {k: sc.transform(v) for k, v in F.items()}
    best = None
    for C in Cs:
        clf = LogisticRegression(C=C, max_iter=3000, class_weight='balanced').fit(F['train'], y_train)
        b = balanced_accuracy_score(y_val, clf.predict(F['val']))
        print(f'  {"+".join(names):<20} C={C:<6} val bal acc {b:.3f}', flush=True)
        if best is None or b > best[0]: best = (b, clf)
    return {k: best[1].predict_proba(F[k]) for k in EVAL}

save_preds('clip_probe_bal', probe(['clip']))
save_preds('siglip_probe', probe(['siglip']))
save_preds('feat_ensemble', probe(['clip', 'dinov2', 'siglip']))
for n in ('clip_probe_bal', 'siglip_probe', 'feat_ensemble'): show(n)

# Option B done properly: a probe trained directly on the 4 groups (not only merged afterwards)
y4 = {k: collapse(Y[k], np.eye(len(CLASSES))[Y[k]], '4 groups')[0] for k in sets}
P4_TRAINED = probe(['clip', 'dinov2', 'siglip'], y4['train'], y4['val'])
print(f'elapsed {(time.time()-T_START)/60:.0f} min')

  clip                 C=0.003  val bal acc 0.651
  clip                 C=0.01   val bal acc 0.655
  clip                 C=0.03   val bal acc 0.659
  clip                 C=0.1    val bal acc 0.655
  siglip               C=0.003  val bal acc 0.684
  siglip               C=0.01   val bal acc 0.682
  siglip               C=0.03   val bal acc 0.677
  siglip               C=0.1    val bal acc 0.671
  clip+dinov2+siglip   C=0.003  val bal acc 0.674
  clip+dinov2+siglip   C=0.01   val bal acc 0.665
  clip+dinov2+siglip   C=0.03   val bal acc 0.652
  clip+dinov2+siglip   C=0.1    val bal acc 0.648
clip_probe_bal        val  65.9  test  72.7  met  68.3  rijks  67.1   (6-style bal acc; museums = unseen artists)
siglip_probe          val  68.4  test  80.6  met  70.9  rijks  70.9   (6-style bal acc; museums = unseen artists)
feat_ensemble         val  67.4  test  79.4  met  72.6  rijks  69.3   (6-style bal acc; museums = unseen artists)
  clip+dinov2+siglip   C=0.003  val bal acc 0.777
  clip+d

## 6. Fine-tune CLIP ViT-L/14 (about 25-30 minutes on a T4)
The frozen lower blocks run without gradients (faster). The best epoch on WikiArt val is saved to Drive; if it is already there from an earlier run, training is skipped.

In [7]:
import math, inspect, torch.nn as nn, torch.nn.functional as Fn
from torchvision import transforms as TT
torch.manual_seed(SEED); np.random.seed(SEED)
MEAN, STD = (0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)
train_tf = TT.Compose([TT.RandomResizedCrop(224, scale=(0.5, 1.0)), TT.RandomHorizontalFlip(),
                       TT.ToTensor(), TT.Normalize(MEAN, STD)])
plain_tf = TT.Compose([TT.Resize((224, 224)), TT.ToTensor(), TT.Normalize(MEAN, STD)])

clip = CLIPModel.from_pretrained(MODELS['clip'])
proc = CLIPProcessor.from_pretrained(MODELS['clip'])
with torch.no_grad():   # start the classifier from the style names (zero-shot weights)
    names = [c.replace(' (Late Renaissance)', '') for c in CLASSES]
    tmpl = ['a painting in the style of {}.', 'a {} painting.', 'an artwork from the {} period.']
    W = []
    for n in names:
        t = proc(text=[x.format(n) for x in tmpl], return_tensors='pt', padding=True)
        e = Fn.normalize(vec(clip.get_text_features(**t)), dim=-1); W.append(Fn.normalize(e.mean(0), dim=0))
    W = torch.stack(W)

class StyleCLIP(nn.Module):
    def __init__(self, clip, W, n_train):
        super().__init__()
        v = clip.vision_model
        self.embeddings, self.pre_ln, self.post_ln = v.embeddings, v.pre_layrnorm, v.post_layernorm
        layers = list(v.encoder.layers)
        self.frozen, self.tuned = nn.ModuleList(layers[:-n_train]), nn.ModuleList(layers[-n_train:])
        self.proj = clip.visual_projection
        self.head = nn.Parameter(W.clone())
        self.log_scale = nn.Parameter(torch.tensor(math.log(30.0)))
    @staticmethod
    def run(layer, h):   # works with old and new versions of transformers
        if 'causal_attention_mask' in inspect.signature(layer.forward).parameters: o = layer(h, None, None)
        else: o = layer(h, None)
        return o[0] if isinstance(o, tuple) else o
    def forward(self, x):
        with torch.no_grad():   # frozen part: no gradients needed
            h = self.pre_ln(self.embeddings(x))
            for layer in self.frozen: h = self.run(layer, h)
        for layer in self.tuned: h = self.run(layer, h)
        f = Fn.normalize(self.proj(self.post_ln(h[:, 0])), dim=-1)
        return self.log_scale.exp() * f @ Fn.normalize(self.head, dim=-1).t()

model = StyleCLIP(clip, W, N_TRAIN_BLOCKS); del clip
for p in model.parameters(): p.requires_grad = False
groups, BASE_LR, DECAY = [], 1e-5, 0.8
for depth, blk in enumerate(reversed(model.tuned)):   # lower blocks get smaller steps
    for p in blk.parameters(): p.requires_grad = True
    groups.append(dict(params=list(blk.parameters()), lr=BASE_LR * DECAY ** depth))
top = list(model.post_ln.parameters()) + list(model.proj.parameters())
for p in top + [model.head, model.log_scale]: p.requires_grad = True
groups += [dict(params=top, lr=BASE_LR), dict(params=[model.head, model.log_scale], lr=1e-3, weight_decay=0.0)]
model = model.to(dev)
trainable = lambda: {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}
ZERO_SHOT = trainable()
print(f'trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.0f} M')

counts = np.bincount(Y['train'], minlength=len(CLASSES))
weights = torch.tensor(len(Y['train']) / (len(CLASSES) * counts), dtype=torch.float32, device=dev)
loss_fn = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)

class Labeled(Dataset):
    def __init__(self, k, tf): self.paths, self.y, self.tf = sets[k]['image'].tolist(), Y[k], tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i): return self.tf(Image.open(DATA / self.paths[i]).convert('RGB')), self.y[i]

@torch.no_grad()
def predict(k, flip=False):
    model.eval(); out = []
    for x, _ in DataLoader(Labeled(k, plain_tf), batch_size=64, num_workers=2):
        x = x.to(dev)
        with torch.autocast('cuda', dtype=torch.float16):
            pr = model(x).float().softmax(-1)
            if flip: pr = (pr + model(torch.flip(x, dims=[3])).float().softmax(-1)) / 2
        out.append(pr.cpu().numpy())
    return np.concatenate(out)

CKPT = OUT / f'clip_ft_seed{SEED}_best.pt'
if CKPT.exists():
    best = (None, torch.load(CKPT)); print('fine-tuned checkpoint found on Drive, skipping training:', CKPT)
else:
    train_dl = DataLoader(Labeled('train', train_tf), batch_size=32, shuffle=True, num_workers=2, drop_last=True)
    EP = 1 if QUICK else FT_EPOCHS
    opt = torch.optim.AdamW(groups, weight_decay=0.05)
    total, warm = EP * len(train_dl), max(1, len(train_dl) // 2)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler('cuda')
    params = [p for g in groups for p in g['params']]
    best, t_ft = (-1, None), time.time()
    for ep in range(EP):
        model.train(); t0, run = time.time(), 0.0
        for i, (x, y) in enumerate(train_dl):
            with torch.autocast('cuda', dtype=torch.float16):
                loss = loss_fn(model(x.to(dev)).float(), y.to(dev))
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); sched.step(); run += loss.item()
            if i == 50 and ep == 0:
                print(f'  speed check: about {(time.time()-t0)/51*len(train_dl)/60:.0f} min per epoch', flush=True)
        v = balanced_accuracy_score(Y['val'], predict('val').argmax(1))
        print(f'epoch {ep+1}/{EP}  loss {run/len(train_dl):.3f}  val bal acc {v:.3f}  ({(time.time()-t0)/60:.1f} min)', flush=True)
        if v > best[0]:
            best = (v, trainable()); torch.save(best[1], CKPT)
        if (time.time() - t_ft) / 60 > TIME_BUDGET_MIN and ep + 1 < EP:
            print(f'time budget of {TIME_BUDGET_MIN} min reached, stopping after epoch {ep+1}'); break
    print(f'best val bal acc {best[0]:.3f}, saved to {CKPT}')
print(f'elapsed {(time.time()-T_START)/60:.0f} min')

config.json:   0%|          | 0.00/4.52k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.71GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/961k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

trainable params: 102 M
  speed check: about 6 min per epoch
epoch 1/4  loss 1.048  val bal acc 0.680  (7.5 min)
epoch 2/4  loss 0.867  val bal acc 0.655  (7.8 min)
epoch 3/4  loss 0.772  val bal acc 0.649  (7.5 min)
epoch 4/4  loss 0.724  val bal acc 0.667  (7.5 min)
best val bal acc 0.680, saved to /content/drive/MyDrive/wikiart_runs/improve/clip_ft_seed1_best.pt
elapsed 55 min


## 7. WiSE-FT blend and scoring (about 6 minutes)
Blend the fine-tuned weights with the original ones; the blend is chosen on WikiArt val. Then score with the mirror-image average.

In [ ]:
def set_alpha(a):
    with torch.no_grad():
        for n, p in model.named_parameters():
            if p.requires_grad: p.copy_(((1 - a) * ZERO_SHOT[n] + a * best[1][n]).to(dev))
alphas = {}
for a in (0.5, 0.75, 1.0):
    set_alpha(a); alphas[a] = balanced_accuracy_score(Y['val'], predict('val').argmax(1))
    print(f'alpha {a:<5} val bal acc {alphas[a]:.3f}', flush=True)
ALPHA = max(alphas, key=alphas.get); set_alpha(ALPHA)
print('chose alpha', ALPHA)
save_preds('clip_finetuned', {k: predict(k, flip=True) for k in EVAL})
show('clip_finetuned')
print(f'elapsed {(time.time()-T_START)/60:.0f} min')

## 8. Final ensemble: fine-tuned CLIP + feature ensemble (weight chosen on WikiArt val)

In [9]:
A, B = RESULTS['clip_finetuned'], RESULTS['feat_ensemble']
ws = {w: balanced_accuracy_score(Y['val'], (w * A['val'] + (1 - w) * B['val']).argmax(1)) for w in (0, .25, .5, .75, 1)}
print({w: round(v, 3) for w, v in ws.items()})
w = max(ws, key=ws.get); print('chose weight', w, 'for the fine-tuned model')
save_preds('final_ensemble', {k: w * A[k] + (1 - w) * B[k] for k in A})
show('final_ensemble')

{0: np.float64(0.674), 0.25: np.float64(0.684), 0.5: np.float64(0.687), 0.75: np.float64(0.693), 1: np.float64(0.675)}
chose weight 0.75 for the fine-tuned model
final_ensemble        val  69.3  test  78.3  met  70.3  rijks  71.0   (6-style bal acc; museums = unseen artists)


## 9. Summary table (paste this back)
All numbers in %. Museum columns are **unseen artists**. `bal` = balanced accuracy, `top1` = plain accuracy, `top3` = true style in the top 3. `4grp` / `3era` = balanced accuracy after merging styles (option, see top). For reference, before this notebook (unseen artists, balanced accuracy, 6 styles): ResNet-50 Met 57.5, Rijks 58.2; CLIP probe (not class-balanced) Met 68.2, Rijks 66.4.

In [ ]:
rows = []
for name, pr in RESULTS.items():
    r = dict(model=name, val_bal=100 * balanced_accuracy_score(Y['val'], pr['val'].argmax(1)))
    for k in ('test', 'met', 'rijks'):
        m = unseen(k); y, p = Y[k][m], pr[k][m]
        r[f'{k} bal'] = 100 * balanced_accuracy_score(y, p.argmax(1))
        r[f'{k} top1'] = 100 * (y == p.argmax(1)).mean()
        r[f'{k} top3'] = 100 * (np.argsort(-p, 1)[:, :3] == y[:, None]).any(1).mean()
        for g, short in (('4 groups', '4grp'), ('3 eras', '3era')):
            yy, pp = collapse(y, p, g); r[f'{k} {short}'] = 100 * balanced_accuracy_score(yy, pp.argmax(1))
    rows.append(r)
tab = pd.DataFrame(rows).round(1)
pd.set_option('display.width', 300); pd.set_option('display.max_columns', 50)
print(tab.to_string(index=False))
tab.to_csv(OUT / f'improve_summary_s{SEED}.csv', index=False)

# 4-group probe trained directly on the merged labels (option B done properly)
line = 'feat_ensemble trained on 4 groups:'
for k in ('test', 'met', 'rijks'):
    m = unseen(k); line += f'  {k} bal {100*balanced_accuracy_score(y4[k][m], P4_TRAINED[k][m].argmax(1)):.1f}'
print('\n' + line)

## 10. Best model (picked on WikiArt val) with painter-level bootstrap intervals
Intervals resample **painters**, not single paintings, because paintings by the same painter are not independent. The paired test compares the best model with the class-balanced CLIP probe on the same paintings.

In [11]:
BEST = max(RESULTS, key=lambda n: balanced_accuracy_score(Y['val'], RESULTS[n]['val'].argmax(1)))
REF = 'clip_probe_bal'
rng = np.random.default_rng(0)
def painter_groups(k, m):
    a = sets[k]['artist'].astype(str).values[m]
    a = np.where(pd.isna(sets[k]['artist'].values[m]), np.arange(m.sum()).astype(str), a)
    return [np.where(a == u)[0] for u in np.unique(a)]
def boot(y, preds, groups, B=1000):
    out = []
    for _ in range(B):
        idx = np.concatenate([groups[j] for j in rng.integers(0, len(groups), len(groups))])
        out.append([balanced_accuracy_score(y[idx], p[idx]) for p in preds])
    return 100 * np.array(out)

lines = [f'Best model on WikiArt val: {BEST}  (compared with {REF})']
for k in ('test', 'met', 'rijks'):
    m = unseen(k); y = Y[k][m]
    for g in ('6 styles', '4 groups', '3 eras'):
        yy, pb = collapse(y, RESULTS[BEST][k][m], g); _, pr_ = collapse(y, RESULTS[REF][k][m], g)
        b = boot(yy, [pb.argmax(1), pr_.argmax(1)], painter_groups(k, m), B=300 if QUICK else 1000)
        d = b[:, 0] - b[:, 1]; obs = 100 * (balanced_accuracy_score(yy, pb.argmax(1)) - balanced_accuracy_score(yy, pr_.argmax(1))); pval = min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))
        lines.append(f'  {k:<6} {g:<9} n={m.sum():>5}  {BEST} {100*balanced_accuracy_score(yy, pb.argmax(1)):5.1f} '
                     f'[{np.percentile(b[:,0],2.5):5.1f}, {np.percentile(b[:,0],97.5):5.1f}]   '
                     f'vs {REF} {100*balanced_accuracy_score(yy, pr_.argmax(1)):5.1f}   '
                     f'gain {obs:+5.1f} [{np.percentile(d,2.5):+5.1f}, {np.percentile(d,97.5):+5.1f}] p = {pval:.3f}')
report = '\n'.join(lines); print(report)
(OUT / f'improve_report_s{SEED}.txt').write_text(tab.to_string(index=False) + '\n\n' + report)

Best model on WikiArt val: final_ensemble  (compared with clip_probe_bal)
  test   6 styles  n= 3857  final_ensemble  78.3 [ 72.3,  83.5]   vs clip_probe_bal  72.7   gain  +5.6 [ +3.2,  +7.7] p = 0.000
  test   4 groups  n= 3857  final_ensemble  81.7 [ 76.7,  85.7]   vs clip_probe_bal  78.2   gain  +3.4 [ +1.4,  +5.8] p = 0.002
  test   3 eras    n= 3857  final_ensemble  92.1 [ 87.7,  94.8]   vs clip_probe_bal  86.8   gain  +5.3 [ +3.2,  +7.2] p = 0.000
  met    6 styles  n=  731  final_ensemble  70.3 [ 66.0,  76.3]   vs clip_probe_bal  68.3   gain  +2.0 [ -2.7,  +7.1] p = 0.310
  met    4 groups  n=  731  final_ensemble  70.4 [ 66.1,  77.4]   vs clip_probe_bal  73.4   gain  -3.0 [ -8.0,  +2.7] p = 0.266
  met    3 eras    n=  731  final_ensemble  87.0 [ 81.0,  91.7]   vs clip_probe_bal  86.0   gain  +1.0 [ -1.1,  +3.3] p = 0.296
  rijks  6 styles  n= 2994  final_ensemble  71.0 [ 66.6,  75.7]   vs clip_probe_bal  67.1   gain  +3.8 [ -2.0,  +8.1] p = 0.216
  rijks  4 groups  n= 2994  fi

2342

## 11. Where the best model still goes wrong (per-style recall and confusion, unseen artists)

In [12]:
short = [c.split(' ')[0][:9] for c in CLASSES]
for k in ('test', 'met', 'rijks'):
    m = unseen(k); y, p = Y[k][m], RESULTS[BEST][k][m].argmax(1)
    rec = recall_score(y, p, average=None, labels=range(len(CLASSES)))
    print(f'\n{k}: recall per style  ' + '  '.join(f'{s} {100*v:.0f}' for s, v in zip(short, rec)))
    cm = confusion_matrix(y, p, labels=range(len(CLASSES)))
    cmp = pd.DataFrame((100 * cm / np.maximum(cm.sum(1, keepdims=True), 1)).round(0).astype(int),
                       index=[f'true {s}' for s in short], columns=short)
    print(cmp.to_string()); cmp.to_csv(OUT / f'confusion_{BEST}_{k}_s{SEED}.csv')
print(f'\nDone in {(time.time()-T_START)/60:.0f} min. Files are in {OUT}')


test: recall per style  Baroque 71  Mannerism 92  Northern 86  Realism 83  Rococo 77  Romantici 61
                Baroque  Mannerism  Northern  Realism  Rococo  Romantici
true Baroque         71         10         0        0      17          1
true Mannerism        7         92         1        0       0          0
true Northern         1         13        86        0       0          0
true Realism          1          0         0       83       1         14
true Rococo          12          2         1        1      77          8
true Romantici        2          1         0       24      12         61

met: recall per style  Baroque 73  Mannerism 88  Northern 89  Realism 61  Rococo 76  Romantici 35
                Baroque  Mannerism  Northern  Realism  Rococo  Romantici
true Baroque         73         10         1        0      14          0
true Mannerism        6         88         6        0       0          0
true Northern         1         10        89        0       0          